In [3]:
!pip install pandas spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import pandas as pd
import spacy
import re
import uuid

def segment_into_clauses(text, nlp):
    if not isinstance(text, str):
        return []

    # Standardize escaped newlines if pandas read them literally
    text = text.replace('\n', '\n')

    # 1. Regex Structural Splitting
    # Targets legal list markers (e.g., "1.", "1.1", "a)", "(a)", "•", "-") at the start of a newline
    pattern = r'(\n\s*(?:[0-9]+(?:\.[0-9]+)*\.?|[a-z]|\([a-z]\)|•|-)\s+)'
    parts = re.split(pattern, text)

    structural_chunks = []
    current_chunk = ""

    for part in parts:
        if re.match(pattern, part):
            if current_chunk.strip():
                structural_chunks.append(current_chunk.strip())
            current_chunk = part.strip() + " "
        else:
            current_chunk += part.strip()

    if current_chunk.strip():
        structural_chunks.append(current_chunk.strip())

    final_clauses = []

    # 2. NLP Sentence Splitting for long, unformatted blocks
    for chunk in structural_chunks:
        # If a chunk exceeds 250 characters, it is likely a dense paragraph requiring NLP splitting
        if len(chunk) > 250:
            doc = nlp(chunk)
            for sent in doc.sents:
                clean_sent = sent.text.strip()
                if len(clean_sent) > 15: # Filter out short noise/artifacts
                    final_clauses.append(clean_sent)
        else:
            if len(chunk) > 15:
                final_clauses.append(chunk)

    return final_clauses

def build_annotation_dataset():
    print("Loading spaCy model...")
    nlp = spacy.load("en_core_web_sm")

    print("Reading documents.csv...")
    df = pd.read_csv("/documents.csv")

    annotation_rows = []

    for index, row in df.iterrows():
        print(f"Segmenting: {row['provider_name']} - {row['document_type']}")
        clauses = segment_into_clauses(row['full_extracted_text'], nlp)

        for clause in clauses:
            annotation_rows.append({
                "clause_id": f"CLS_{uuid.uuid4().hex[:8].upper()}",
                "document_id": row['document_id'],
                "provider_name": row['provider_name'],
                "document_type": row['document_type'],
                "clause_text": clause,
                "violation_category": "",     # To be filled by annotator
                "risk_level": "",             # To be filled by annotator (0, 1, 2)
                "violating_span": "",         # To be filled by annotator
                "primary_law_reference": "",  # To be filled by annotator
                "annotator_notes": ""         # To be filled by annotator
            })

    out_df = pd.DataFrame(annotation_rows)
    out_df.to_csv("annotation_dataset.csv", index=False)
    print(f"\nSuccess! Extracted {len(out_df)} clauses ready for annotation.")

if __name__ == "__main__":
    build_annotation_dataset()

Loading spaCy model...
Reading documents.csv...
Segmenting: Tala - terms_and_conditions
Segmenting: Tala - privacy_policy
Segmenting: Branch - terms_of_use
Segmenting: Branch - privacy_policy
Segmenting: Zenka - terms_of_use
Segmenting: Zenka - terms_and_conditions
Segmenting: Zenka - privacy_policy
Segmenting: Lendplus - terms_and_conditions
Segmenting: Lendplus - privacy_policy
Segmenting: Berry - terms_and_conditions
Segmenting: Berry - privacy_policy
Segmenting: Mkopa - terms_of_use
Segmenting: Mkopa - privacy_policy
Segmenting: Zanifu - terms_and_conditions
Segmenting: Asante - terms_and_conditions
Segmenting: Asante - privacy_policy
Segmenting: Jumo - privacy_policy
Segmenting: Littlepesa - terms_and_conditions
Segmenting: Littlepesa - privacy_policy
Segmenting: Pezesha - privacy_policy
Segmenting: 4G Capital - terms_and_conditions
Segmenting: Ngao Credit - terms_and_conditions
Segmenting: Longitude Finance - terms_and_conditions
Segmenting: Getcash Capital - terms_and_conditions